## Section 1: System Preparation — App Initialisation & Pre-trained Artefacts

Before the Gradio application can serve any request, a set of heavyweight artefacts must be
prepared **offline** and loaded into memory once at startup.

**Why offline?**
- The GloVe embedding file (`glove.6B.300d.txt`) is ~1 GB and contains 400 K word vectors.
- The review dataset has 61,275 rows; training three classifiers on it takes several minutes.
- Doing either of these at request time would make the app unusable.

**Solution:** `scripts/train_models.py` is run **once** before deploying the app.
It produces **6 pickle files** saved to `models/`:

| File | Contents |
|---|---|
| `model_a_bow.pkl` | MLP classifier (256→128) using unweighted mean GloVe vectors of review text only |
| `model_b_bow_meta.pkl` | MLP classifier using Bag-of-Words over review text + title (CountVectorizer) |
| `model_c_glove_meta.pkl` | Random Forest (200 trees) using unweighted GloVe for text+title + numerical metadata (price, avg rating, rating count, review rating, brand) |
| `label_encoder.pkl` | `sklearn.LabelEncoder` fitted on the 11 brand names — converts brand strings to integers for Model C |
| `collocation_dict.pkl` | 111 domain-specific bigrams (e.g. `"dry skin"` → `"dry-skin"`) discovered by PMI over the corpus and merged with 54 manual patterns |
| `product_vectors.pkl` | 295 product-level 300-dim GloVe vectors, each the mean of its reviews' mean word vectors — used by the recommendation engine |

All three classifiers are trained with **SMOTE** to handle the 78.7 % / 21.3 % buyer/non-buyer
class imbalance, and evaluated on a held-out 20 % stratified split.

#### Step 1: Load Raw Data Assets

We load four input files that the entire training pipeline depends on:
- `processed.csv` — 61,275 reviews with cleaned text from Milestone 1
- `vocab.txt` — 7,366-word vocabulary (word → index mapping)
- `glove.6B.300d.txt` — pre-trained GloVe embeddings; only the vocab-subset is loaded to save memory
- `stopwords_en.txt` — 570 stopwords used in the preprocessing pipeline

In [ ]:
import os, pickle, re
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.neural_network import MLPClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.over_sampling import SMOTE
from nltk.stem import WordNetLemmatizer

from core.preprocessing import (
    UnweightedVectorTransformer,
    preprocess_text,
    load_stopwords,
    build_collocation_dict_from_corpus,
)

# Reviews dataset (61,275 rows after dropping 9 with missing processed text)
df = pd.read_csv('notebooks/processed.csv')
df = df[df['processed_review_text'].notna()]

# Vocabulary: word → index  (7,366 words from Milestone 1 pipeline)
vocab_dict = {}
with open('notebooks/vocab.txt', encoding='utf-8') as f:
    for line in f:
        if ':' in line:
            word, idx = line.strip().rsplit(':', 1)
            vocab_dict[word] = int(idx)
vocab_set = set(vocab_dict.keys())

# GloVe: load only vocab-subset vectors from the full 400 K-word file
embeddings_dict = {}
with open('notebooks/glove.6B.300d.txt', encoding='utf-8') as f:
    for line in f:
        parts = line.strip().split()
        if parts[0] in vocab_set:
            embeddings_dict[parts[0]] = np.array(parts[1:], dtype=np.float32)

stopwords = load_stopwords('notebooks/stopwords_en.txt')

#### Step 2: Build Collocation Dictionary via PMI

**Pointwise Mutual Information (PMI)** measures how much more often two words appear
together than we would expect by chance. A high PMI score for "matte" + "finish" means
they co-occur far more than random — indicating a meaningful domain phrase.

We scan all 61,275 reviews, keep bigrams with frequency ≥ 5 and PMI ≥ 3.0,
then merge with 54 manually curated patterns from Milestone 1.
The result is ~111 bigrams (e.g. `"dry skin"` → `"dry-skin"`) that are treated as
single tokens throughout the pipeline.

In [ ]:
colloc_dict = build_collocation_dict_from_corpus(
    df['review_text'].fillna(''),
    vocab_set,
    min_freq=5,
    pmi_threshold=3.0
)
print(f"Built {len(colloc_dict)} collocations")

#### Step 3: Preprocess Review Titles

Review titles are preprocessed through the same 5-step pipeline as review bodies
(see Section 2a) so they can be used as a separate text feature in Models B and C.

In [ ]:
lemmatizer = WordNetLemmatizer()
df['processed_title'] = df['review_title'].fillna('').apply(
    lambda x: preprocess_text(x, colloc_dict, stopwords, lemmatizer)
)

#### Step 4: Encode Brand Names

Model C uses brand name as a numerical metadata feature.
`LabelEncoder` maps each of the 11 brand names to an integer (0–10).
Unseen brands at inference time fall back to 0.

In [ ]:
le = LabelEncoder()
df['brand_encoded'] = le.fit_transform(df['brand_name'].fillna('unknown'))
print(f"Encoded {len(le.classes_)} brands: {list(le.classes_)}")

#### Step 5: Stratified Train/Test Split with SMOTE

The dataset is heavily **imbalanced**: 78.7 % buyers vs 21.3 % non-buyers.
Two strategies address this:

- **Stratified split** — ensures both train and test sets preserve the same 78.7/21.3 ratio,
  so evaluation metrics are not distorted.
- **SMOTE** (Synthetic Minority Over-sampling Technique) — generates synthetic non-buyer
  examples during training so the classifier does not simply learn to predict "buyer" for
  everything. Applied inside the pipeline after splitting, so test data is never touched.

In [ ]:
df['is_a_buyer'] = df['is_a_buyer'].map({True:1, False:0, 'True':1, 'False':0, 1:1, 0:0})
y = df['is_a_buyer'].astype(int)

X_train, X_test, y_train, y_test = train_test_split(
    df, y, test_size=0.2, random_state=42, stratify=y
)
print(f"Train: {len(X_train):,}  |  Test: {len(X_test):,}")
print(f"Buyer rate — train: {y_train.mean():.1%}  test: {y_test.mean():.1%}")

#### Step 6: Train Model A — MLP + Unweighted GloVe (Text Only)

**Input:** preprocessed review text only
**Representation:** 300-dim unweighted mean of GloVe word vectors
**Classifier:** Multi-Layer Perceptron with hidden layers (256, 128)

This model captures the semantic content of the review body without any metadata,
making it a pure text signal for the ensemble.

In [ ]:
model_a = ImbPipeline([
    ('vec',   UnweightedVectorTransformer(embeddings_dict)),
    ('smote', SMOTE(random_state=42)),
    ('clf',   MLPClassifier(hidden_layer_sizes=(256, 128), max_iter=200,
                             random_state=42, early_stopping=True))
])
model_a.fit(X_train['processed_review_text'], y_train)

#### Step 7: Train Model B — MLP + Bag-of-Words (Text + Title)

**Input:** preprocessed review text + review title
**Representation:** sparse Bag-of-Words vectors via `CountVectorizer`
  - Review body: fixed vocabulary of 7,366 words from Milestone 1
  - Title: open vocabulary fitted on the training titles
**Classifier:** Multi-Layer Perceptron with hidden layers (256, 128)

Using BoW captures exact word frequencies; combining title gives the model a secondary
signal — reviewers who are buyers often write more specific, product-focused titles.

In [ ]:
meta_cols = ['price', 'avg_product_rating', 'product_rating_count',
             'review_rating', 'brand_encoded']

model_b = ImbPipeline([
    ('features', ColumnTransformer([
        ('text',  CountVectorizer(vocabulary=vocab_dict), 'processed_review_text'),
        ('title', CountVectorizer(),                      'processed_title'),
    ])),
    ('smote', SMOTE(random_state=42)),
    ('clf',   MLPClassifier(hidden_layer_sizes=(256, 128), max_iter=200,
                             random_state=42, early_stopping=True))
])
model_b.fit(X_train, y_train)

#### Step 8: Train Model C — Random Forest + GloVe + Metadata

**Input:** preprocessed review text + title + 5 numerical features
**Representation:**
  - Text (review body): 300-dim unweighted GloVe mean
  - Title: 300-dim unweighted GloVe mean
  - Metadata: price, avg product rating, product rating count, review rating, brand (encoded) — imputed and standardised
**Classifier:** Random Forest with 200 trees

This is the richest model. Random Forest handles the mixed dense+numerical feature space
well and is robust to correlated features without needing feature scaling for the tree splits
(though we still scale for consistency).

In [ ]:
model_c = ImbPipeline([
    ('features', ColumnTransformer([
        ('text',  UnweightedVectorTransformer(embeddings_dict), 'processed_review_text'),
        ('title', UnweightedVectorTransformer(embeddings_dict), 'processed_title'),
        ('meta',  Pipeline([
            ('imputer', SimpleImputer(strategy='constant', fill_value=0)),
            ('scaler',  StandardScaler())
        ]), meta_cols)
    ])),
    ('smote', SMOTE(random_state=42)),
    ('clf',   RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1))
])
model_c.fit(X_train, y_train)

#### Step 9: Compute Product-Level GloVe Vectors

For the recommendation engine (Task 3), we need a single 300-dim vector per product
that summarises what reviewers say about it.

**Method:** for each product, average the per-review mean GloVe vectors across all its reviews.
This gives a centroid in semantic space that represents the "typical language" used for that product.
Products with similar review language will end up close together in this 300-dim space.

In [ ]:
product_vectors = {}
for pid, group in df.groupby('product_id'):
    vecs = []
    for text in group['processed_review_text']:
        tokens    = str(text).split()
        word_vecs = [embeddings_dict[w] for w in tokens if w in embeddings_dict]
        if word_vecs:
            vecs.append(np.mean(word_vecs, axis=0))
    product_vectors[pid] = np.mean(vecs, axis=0) if vecs else np.zeros(300)

print(f"Computed vectors for {len(product_vectors)} products")

#### Step 10: Save All Artefacts

All 6 objects are serialised with `pickle` to `models/`.
At app startup, `data_loader.load_all()` deserialises them in a single call — no retraining needed.

In [ ]:
os.makedirs('models', exist_ok=True)
for path, obj in [
    ('models/model_a_bow.pkl',        model_a),
    ('models/model_b_bow_meta.pkl',   model_b),
    ('models/model_c_glove_meta.pkl', model_c),
    ('models/label_encoder.pkl',      le),
    ('models/collocation_dict.pkl',   colloc_dict),
    ('models/product_vectors.pkl',    product_vectors),
]:
    with open(path, 'wb') as f:
        pickle.dump(obj, f)
    print(f"Saved {path}  ({os.path.getsize(path)/1e6:.1f} MB)")

### App Startup — `core/data_loader.load_all()`

When `app.py` launches, it calls a single function `load_all()` which:

1. Reads `notebooks/processed.csv` (61,275 reviews)
2. Loads `vocab.txt` (7,366 words → index mapping)
3. Loads `stopwords_en.txt` (570 words)
4. Loads full GloVe embeddings — parses `glove.6B.300d.txt` on first run and caches the result
   to `models/glove_cache.pkl`; subsequent startups load from the cache (~10× faster)
5. Deserialises the 6 pickle files above
6. Pre-computes `products_df` — one row per product with aggregated fields
   (brand, title, price, avg rating, review count, buyer count)
7. Builds `reviews_by_product` — a `dict[product_id → list[review_dict]]` for O(1) lookup

Everything is returned as a single dictionary and injected into every UI tab.

In [ ]:
import os, pickle
import numpy as np
import pandas as pd

def load_all(base_path='.'):
    """Load all data assets once at app startup."""

    # 1. Reviews dataset
    df = pd.read_csv(os.path.join(base_path, 'notebooks', 'processed.csv'))

    # 2. Vocabulary  (word → index)
    vocab = {}
    with open(os.path.join(base_path, 'notebooks', 'vocab.txt'), encoding='utf-8') as f:
        for line in f:
            if ':' in line:
                word, idx = line.strip().rsplit(':', 1)
                vocab[word] = int(idx)

    # 3. Stopwords
    with open(os.path.join(base_path, 'notebooks', 'stopwords_en.txt'), encoding='utf-8') as f:
        stopwords = {line.strip().lower() for line in f if line.strip()}

    # 4. GloVe embeddings — parse once then cache as pickle for fast reloads
    cache_path = os.path.join(base_path, 'models', 'glove_cache.pkl')
    if os.path.exists(cache_path):
        with open(cache_path, 'rb') as f:
            glove_embeddings = pickle.load(f)
    else:
        glove_embeddings = {}
        with open(os.path.join(base_path, 'notebooks', 'glove.6B.300d.txt'), encoding='utf-8') as f:
            for line in f:
                parts = line.strip().split()
                glove_embeddings[parts[0]] = np.array(parts[1:], dtype=np.float32)
        with open(cache_path, 'wb') as f:
            pickle.dump(glove_embeddings, f, protocol=4)

    # 5. Load the 6 pickled model artefacts
    models_dir = os.path.join(base_path, 'models')
    models = {}
    for fname in ['model_a_bow.pkl', 'model_b_bow_meta.pkl', 'model_c_glove_meta.pkl',
                  'label_encoder.pkl', 'collocation_dict.pkl', 'product_vectors.pkl']:
        with open(os.path.join(models_dir, fname), 'rb') as f:
            models[fname.replace('.pkl', '')] = pickle.load(f)

    # 6. Product-level aggregations (one row per product)
    products_df = df.groupby('product_id').agg(
        brand_name=('brand_name', 'first'),
        product_title=('product_title', 'first'),
        price=('price', 'first'),
        avg_product_rating=('avg_product_rating', 'first'),
        product_rating_count=('product_rating_count', 'first'),
        product_url=('product_url', 'first'),
        review_count=('review_id', 'count')
    ).reset_index()
    buyer_counts = (
        df[df['is_a_buyer'].isin([True, 1, 'True'])]
        .groupby('product_id').size().reset_index(name='buyer_count')
    )
    products_df = products_df.merge(buyer_counts, on='product_id', how='left')
    products_df['buyer_count'] = products_df['buyer_count'].fillna(0).astype(int)

    # 7. Reviews-by-product index  →  O(1) lookup at request time
    reviews_by_product = {
        pid: group.to_dict('records')
        for pid, group in df.groupby('product_id')
    }

    return dict(df=df, vocab=vocab, stopwords=stopwords,
                glove_embeddings=glove_embeddings, models=models,
                products_df=products_df, reviews_by_product=reviews_by_product)